# Ice: ventas de videojuegos por región

**Brenis Hernández · Proyecto de portafolio**

Esta copia pública conserva el código y las explicaciones. Se retiraron las salidas originales para evitar publicar registros o identificadores; los resultados resumidos se documentan en la ficha del portafolio a partir de la versión original revisada. No se ha vuelto a ejecutar sobre los datos originales. También se omiten mensajes de revisión académica.

**Alcance:** El objetivo original es planear campañas para 2017; los datos no describen el mercado actual. Ventas no equivalen a rentabilidad, y una correlación entre reseñas y ventas no demuestra causalidad.


----

# Introducción

Trabajamos como analistas para Ice, una tienda online que vende videojuegos en todo el mundo.  
Nuestro objetivo es identificar patrones que determinen si un juego tendrá éxito, para poder planificar campañas publicitarias en 2017.


# Pasos a seguir

1. Carga y revisión general de los datos. 
   Exploraremos la estructura, tipos de datos y contenido inicial.  

2. Preparación del dataset.  
   - Normalización de columnas a minúsculas.  
   - Conversión de tipos numéricos.  
   - Tratamiento de valores ausentes (`tbd` → NaN, años faltantes, etc.).  
   - Creación de la columna `global_sales`.
   - Valores ausentes

3. Análisis exploratorio de datos (EDA). 
   - Juegos lanzados por año.  
   - Ventas globales por plataforma y tendencias.  
   - Selección de ventana temporal adecuada para analizar 2017.  
   - Boxplots de ventas por plataforma.  
   - Relación entre reseñas (críticos/usuarios) y ventas.  
   - Ventas por género.
   - Varianza y desviación estándar  

4. Perfil de usuario por región. 
   - Las 5 plataformas principales.
   - Los 5 géneros principales. 
   - Análisis del impacto del rating ESRB en cada región.  

5. Pruebas de hipótesis estadísticas.  
   - H1: medias de `user_score` iguales para XOne y PC.  
   - H2: medias de `user_score` diferentes para Action y Sports.  
   - Establecer hipótesis nula/alternativa, nivel α = 0.05, método t-test de Welch.  

6. Conclusión general. 
   Resumiremos hallazgos, implicaciones estratégicas y recomendaciones para 2017.



# Inicializacion

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats as st

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Paso 1. Abrir archivo de datos y estudia la informacion general

In [ ]:
path_main = Path("/datasets/games.csv")

df = pd.read_csv(path_main)
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T.head(30)

# Paso 2. Prepara los datos

2.1 Reemplaza los nombres de las columnas (ponlos en minúsculas).

In [ ]:
df.columns = [c.strip().lower() for c in df.columns]

for col in ["name", "platform", "genre", "rating"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()


2.2 Convierte los datos en los tipos necesarios.
2.3 Manejo de tbd.

In [ ]:
orig_dtypes = df.dtypes.astype(str)

# Años y puntuaciones
df["year_of_release"] = pd.to_numeric(df["year_of_release"], errors="coerce")
df["critic_score"] = pd.to_numeric(df["critic_score"], errors="coerce")

# 'user_score' viene como texto y puede traer 'tbd'
df["user_score"] = (
    df["user_score"]
      .replace({"tbd": np.nan, "TBD": np.nan})
      .replace(r"^\s*$", np.nan, regex=True)
)
df["user_score"] = pd.to_numeric(df["user_score"], errors="coerce")

# Ventas por región (a millones USD)
for col in ["na_sales", "eu_sales", "jp_sales", "other_sales"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

2.4 Crear global sales y revisar cambios de tipos

In [ ]:
df["global_sales"] = (
    df["na_sales"].fillna(0)
  + df["eu_sales"].fillna(0)
  + df["jp_sales"].fillna(0)
  + df["other_sales"].fillna(0)
)

new_dtypes = df.dtypes.astype(str)
dtype_changes = (
    pd.DataFrame({"original": orig_dtypes, "nuevo": new_dtypes})
      .query("original != nuevo")
)
dtype_changes


2.5 Valores ausentes

In [ ]:
missing_pct = df.isna().mean().mul(100).round(2).sort_values(ascending=False)
missing_pct


Decisiones de tratamiento de ausentes
- `user_score`: la cadena TBD significa “por determinar” → se deja como NaN (no inventar puntuaciones).  
- Ventas por región: si falta en alguna región, asumimos 0 para el cálculo de `global_sales` (suma de regiones).  
- `year_of_release`: si es NaN, el registro no participa en análisis por año/tendencias.


Posibles causas de valores ausentes
- Datos incompletos en fuentes abiertas: no siempre se tienen reseñas de usuarios o críticos para todos los juegos.  
- Lanzamientos recientes (2016): algunos datos pudieron no estar disponibles todavía al momento del corte.  
- Regiones específicas: si un juego nunca se distribuyó en cierta región, esa columna queda en blanco (equivalente a cero ventas).  
- Errores de carga / registro: también es posible que algunas entradas se hayan perdido en la recolección original.

# Paso 3. Analiza los datos

3.1 Mira cuántos juegos fueron lanzados en diferentes años.

In [ ]:
games_by_year = (
    df.dropna(subset=["year_of_release"])
      .groupby("year_of_release")["name"]
      .nunique()
      .sort_index()
)

ax = games_by_year.plot(kind="bar", figsize=(12,4))
ax.set_title("Número de juegos lanzados por año")
ax.set_xlabel("Año de lanzamiento")
ax.set_ylabel("Conteo de juegos")
plt.tight_layout(); plt.show()

games_by_year.tail(10)


¿Son significativos los datos de cada período?

- Los datos previos a 2000 no son representativos para el análisis.  
- Para planificar campañas en 2017, lo más significativo es concentrarse en la ventana **2013–2016**, que refleja el mercado actual y las plataformas vigentes.


3.2 Plataformas con mayores ventas totales.


In [ ]:
platform_sales_total = (
    df.groupby("platform")["global_sales"]
      .sum()
      .sort_values(ascending=False)
)
platform_sales_total.head(10)



 Tendencias por ano en plataformas top


In [ ]:
top_platforms = platform_sales_total.head(6).index.tolist()

plat_year_sales = (
    df[df["platform"].isin(top_platforms)]
      .dropna(subset=["year_of_release"])
      .groupby(["year_of_release","platform"])["global_sales"]
      .sum()
      .unstack("platform")
      .fillna(0)
      .sort_index()
)

plt.figure(figsize=(12,5))
for p in plat_year_sales.columns:
    plt.plot(plat_year_sales.index, plat_year_sales[p], label=p)
plt.title("Ventas globales por año (plataformas top)")
plt.xlabel("Año")
plt.ylabel("Ventas globales (millones)")
plt.legend()
plt.tight_layout(); plt.show()

plat_year_sales.tail(10)


¿Cuánto tardan generalmente las nuevas plataformas en aparecer y las antiguas en desaparecer?

En el gráfico se observa que plataformas como **PS2, Wii, DS, Xbox360** fueron muy populares en su momento (entre 2005 y 2012), pero actualmente ya no tienen ventas, lo que refleja el final de su ciclo de vida.
- Nuevas plataformas como **PS4 y XOne** aparecen alrededor de 2013–2014 y tardan 1–2 años en alcanzar un volumen de ventas considerable.
- El tiempo de “desaparición” de las consolas antiguas también es de aproximadamente 2–3 años desde que sale la nueva generación. Por ejemplo, PS3 y Xbox360 siguieron teniendo ventas hasta 2014, pero fueron cayendo rápidamente después.
- En el caso de portátiles, 3DS mantuvo relevancia en Japón hasta fechas más recientes, mientras que la PSP ya prácticamente desapareció.

**Conclusión:**  
El ciclo típico de las plataformas es de unos 10 años en total (5 a 6 años de auge + 2 a 3 de declive). Esto implica que para 2017 las plataformas más relevantes a considerar son **PS4, XOne y 3DS**, mientras que PS3 y Xbox360 ya no son estratégicas.

3.3  Elegir ventana reciente para modelar 2017

In [ ]:
last_year = int(df["year_of_release"].dropna().max())
analysis_start_year = max(2013, last_year - 3)  # últimos 4 años disponibles hasta 2016
analysis_start_year, last_year


In [ ]:
recent = df[
    (df["year_of_release"] >= analysis_start_year) &
    (df["year_of_release"] <= last_year)
].copy()
recent.shape



3.4 Plataformas líderes y distribución de ventas (ventana reciente)

In [ ]:
recent_top_platforms = (
    recent.groupby("platform")["global_sales"]
          .sum()
          .sort_values(ascending=False)
          .head(6)
          .index.tolist()
)
recent_top_platforms


In [ ]:
box_data = [recent.loc[recent["platform"]==p, "global_sales"].dropna()
            for p in recent_top_platforms]

plt.figure(figsize=(12,5))
plt.boxplot(box_data, labels=recent_top_platforms, showfliers=False)
plt.title(f"Distribución de ventas globales por plataforma ({analysis_start_year}-{last_year})")
plt.xlabel("Plataforma")
plt.ylabel("Ventas por juego (millones)")
plt.tight_layout(); plt.show()


¿Son significativas las diferencias en las ventas? ¿Qué sucede con las ventas promedio en varias plataformas? Describe tus hallazgos.


- El diagrama de cajas muestra que sí existen diferencias significativas entre plataformas.
- **PS4** presenta la mayor mediana de ventas por juego y también la mayor dispersión: tiene varios títulos con ventas muy altas (outliers → “megahits”).
- **XOne** muestra un patrón similar, pero con ventas promedio algo menores.
- **3DS** tiene ventas más modestas y consistentes: la mediana es baja, aunque algunos títulos llegan a ser exitosos en Japón.
- Plataformas en declive (como PS3 y X360) tienen ventas promedio muy bajas en este periodo y poca dispersión, lo que confirma que ya no son competitivas.

Conclusión:  
- Las diferencias entre plataformas son claras: PS4 lidera, seguida de XOne, mientras que 3DS ocupa un nicho específico.  
- Las ventas promedio de PS4 y XOne superan por mucho a las de otras plataformas, lo que las convierte en las opciones prioritarias para campañas publicitarias en 2017.




3.5 Relación entre reseñas (críticos/usuarios) y ventas.

In [ ]:

chosen = (
    recent.groupby("platform")["global_sales"]
          .sum()
          .idxmax()
)
chosen




In [ ]:

sub = df[df["platform"]==chosen].copy()

# Correlaciones de Pearson (usa nan_policy omit implícito al filtrar)
mask_c = sub["critic_score"].notna() & sub["global_sales"].notna()
mask_u = sub["user_score"].notna() & sub["global_sales"].notna()

corr_critic = st.pearsonr(sub.loc[mask_c,"critic_score"], sub.loc[mask_c,"global_sales"]).statistic if mask_c.sum()>=2 else np.nan
corr_user   = st.pearsonr(sub.loc[mask_u,"user_score"],   sub.loc[mask_u,"global_sales"]).statistic if mask_u.sum()>=2 else np.nan

corr_critic, corr_user



In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(sub["critic_score"], sub["global_sales"])
plt.title(f"{chosen}: Ventas vs Critic_Score")
plt.xlabel("Critic_Score (0-100)")
plt.ylabel("Ventas globales (millones)")
plt.tight_layout(); plt.show()

plt.figure(figsize=(6,5))
plt.scatter(sub["user_score"], sub["global_sales"])
plt.title(f"{chosen}: Ventas vs User_Score")
plt.xlabel("User_Score (0-10)")
plt.ylabel("Ventas globales (millones)")
plt.tight_layout(); plt.show()



Mira cómo las reseñas de usuarios y profesionales afectan las ventas de una plataforma popular (tu elección). Crea un gráfico de dispersión y calcula la correlación entre las reseñas y las ventas. Saca conclusiones.


- El gráfico de dispersión **Critic_Score vs ventas** muestra una tendencia ascendente: cuanto mejor la puntuación de críticos, mayores las ventas del juego.  
  - La correlación es **≈ 0.41**, lo que indica una relación positiva moderada.
- El gráfico de dispersión **User_Score vs ventas** no muestra un patrón claro: los puntos están muy dispersos.  
  - La correlación es **≈ -0.03**, lo que significa que prácticamente **no hay relación** entre la puntuación de usuarios y las ventas.

Conclusión:  
Las **reseñas de críticos influyen de forma significativa en las ventas de los juegos de PS4**, mientras que las reseñas de usuarios parecen no tener impacto.  
Esto sugiere que, al planificar campañas, la cobertura en prensa y portales especializados puede ser más determinante que la opinión de los jugadores en línea.



**Teniendo en cuenta tus conclusiones compara las ventas de los mismos juegos en otras plataformas.**

Al revisar los títulos multiplataforma más vendidos (ejemplo: FIFA, Call of Duty, GTA V):

- Generalmente, la plataforma líder concentra la mayor parte de las ventas.  
  - Ejemplo: en títulos multiplataforma recientes, **PS4** acumula más ventas que XOne o PC.  
- **XOne** suele ser la segunda en ventas, aunque en proporción menor.  
- **PC** y algunas portátiles (3DS, PSV) tienen participaciones mucho más bajas para los mismos títulos.

Conclusión: 
Incluso cuando un juego está disponible en varias plataformas, la **plataforma con mayor base de usuarios activa (PS4 en este periodo)** logra captar la mayoría de las ventas.  
Esto refuerza la idea de priorizar campañas en PS4 y XOne, ya que allí se concentra la demanda de títulos multiplataforma.



3.6 Distribución por género

In [ ]:
genre_sales = (
    df.groupby("genre")["global_sales"]
      .agg(total_sales="sum", avg_sales="mean", count_games="size")
      .sort_values("total_sales", ascending=False)
)
genre_sales.head(10)


In [ ]:
ax = (df.groupby("genre")["global_sales"].sum()
        .sort_values(ascending=False)
        .plot(kind="bar", figsize=(12,5)))
ax.set_title("Ventas globales por género")
ax.set_xlabel("Género")
ax.set_ylabel("Ventas globales (millones)")
plt.tight_layout(); plt.show()


¿Qué se puede decir de los géneros más rentables? ¿Puedes generalizar acerca de los géneros con ventas altas y bajas?

- Los géneros con **mayores ventas globales** son **Action**, **Shooter** y **Sports**, que concentran la mayor parte del mercado. Estos géneros suelen incluir franquicias muy populares (*Call of Duty, FIFA, Grand Theft Auto*), lo que explica su volumen de ventas.
- Otros géneros como **Role-Playing (RPG)** y **Platform** también alcanzan cifras altas, aunque en mercados más específicos (ej. Japón con RPG).
- En contraste, géneros como **Puzzle**, **Adventure** o **Strategy** presentan **ventas mucho más bajas**, lo que indica que son nichos reducidos o con menos impacto comercial global.

Conclusión:  
Se puede generalizar que los géneros **de acción y deporte son los más rentables a nivel mundial**, mientras que los géneros **más casuales o de nicho tienen un peso menor**.  
Esto sugiere que para campañas de 2017 conviene enfocar esfuerzos en Action, Shooter y Sports, aunque sin dejar de lado géneros estratégicos en regiones específicas (ej. RPG en Japón).


3.7  Varianza y desviación estándar

In [ ]:
# Desviación estándar y varianza de ventas globales (todos los juegos)
global_std = df["global_sales"].std()
global_var = df["global_sales"].var()

print(f"Desviación estándar global: {global_std:.2f}")
print(f"Varianza global: {global_var:.2f}")


In [ ]:
# Desviación estándar y varianza por plataforma (ventana reciente)
stats_platform = (
    recent.groupby("platform")["global_sales"]
          .agg(["mean","std","var","count"])
          .sort_values("mean", ascending=False)
)
stats_platform.head(10)


In [ ]:
# Desviación estándar y varianza por género
stats_genre = (
    df.groupby("genre")["global_sales"]
      .agg(["mean","std","var","count"])
      .sort_values("mean", ascending=False)
)
stats_genre.head(10)


- La **varianza** mide la dispersión de las ventas respecto a la media, en unidades al cuadrado.  
- La **desviación estándar** es su raíz cuadrada → más intuitiva porque está en las mismas unidades que las ventas (millones de USD).  
- Valores altos en plataformas como PS4 o géneros como Action muestran que existen pocos títulos con ventas enormes y muchos con ventas bajas.  
- Géneros/plataformas con varianza baja (ej. Puzzle en JP) indican resultados más consistentes, aunque no necesariamente grandes.

# Paso 4. Crea un perfil de usuario para cada región

4.1 Las 5 plataformas principales.

In [ ]:
def top5_platforms_share(table, region_col):
    sales = table.groupby("platform")[region_col].sum().sort_values(ascending=False)
    share = (sales / sales.sum()) * 100
    top5 = sales.head(5)
    return pd.DataFrame({
        "sales_musd": top5.round(2),
        "share_%": share.loc[top5.index].round(2)
    })

for r in ["na_sales","eu_sales","jp_sales"]:
    print(f"\nTop 5 plataformas en {r.upper()}")
    display(top5_platforms_share(df, r))


4.2 Los cinco generos principales.

In [ ]:
for r in ["na_sales","eu_sales","jp_sales"]:
    print(f"\nTop 5 géneros en {r.upper()}")
    display(df.groupby("genre")[r].sum().sort_values(ascending=False).head(5).round(2))




4.3 ESRB: participación por rating en cada región

In [ ]:
df["rating_filled"] = df["rating"].replace({"nan": np.nan}).fillna("Unknown")

for r in ["na_sales","eu_sales","jp_sales"]:
    share = (df.groupby("rating_filled")[r].sum() / df[r].sum() * 100).round(2)
    print(f"\nParticipación por rating ESRB en {r.upper()}")
    display(share.sort_values(ascending=False))


**Conclusiones**

Norteamérica (NA)
- Plataformas top:predominan consolas de sobremesa como **X360, PS4, PS3 y XOne**, que concentran la mayor parte de las ventas.  
- Géneros top: **Action, Shooter y Sports** dominan, reflejando la preferencia por títulos competitivos y de acción rápida.  
- ESRB: las clasificaciones **Mature (M)** y **Teen (T)** concentran la mayor parte de las ventas. Esto indica que los jugadores en NA consumen títulos con contenido más adulto.

Europa (EU)
- Plataformas top: destacan **PS3, PS4, X360 y XOne**, con una distribución similar a NA pero con mayor peso de PlayStation.  
- Géneros top: otra vez **Action, Shooter y Sports**, aunque con algo más de equilibrio con géneros como Racing.  
- ESRB: también predominan las clasificaciones **Mature** y **Teen**, con un patrón muy parecido al de NA.

Japón (JP)
- Plataformas top: aquí cambia el panorama: las portátiles como **3DS, DS y PSP** lideran, junto con **PS3**. El mercado japonés favorece consolas portátiles y de Sony.  
- Géneros top: a diferencia de NA/EU, en Japón los más vendidos son **Role-Playing (RPG)** y **Action**, confirmando la popularidad de franquicias japonesas de RPG.  
- ESRB: la participación del rating es mucho menor y hay un alto porcentaje de juegos sin clasificación registrada (**Unknown**). Esto sugiere que el impacto de ESRB en JP es limitado.

Conclusión regional
- NA y EU son mercados similares → dominan consolas de sobremesa (PlayStation y Xbox), géneros Action/Shooter/Sports, y ratings Mature/Teen.  
- JP es distinto → se concentra en portátiles y RPG, con menor relevancia del rating ESRB.  
- Esto implica que las campañas publicitarias deben adaptarse **a cada región**:  
  - Occidente (NA/EU): enfocar en PS4/XOne y géneros de acción y deportes, destacando la clasificación ESRB.  
  - Japón: priorizar 3DS y RPG, donde la clasificación ESRB no influye tanto.


# Paso 5. Prueba las siguientes hipótesis:

Hipótesis 1. Las calificaciones promedio de los usuarios para las plataformas Xbox One y PC son las mismas.
Hipótesis 2. Las calificaciones promedio de los usuarios para los géneros de Acción y Deportes son diferentes.

In [ ]:
alpha = 0.05

# H1: medias de user_score para XOne y PC son iguales (H0: μ_XOne = μ_PC; H1: μ_XOne ≠ μ_PC)
xone = df.loc[(df["platform"]=="XOne") & df["user_score"].notna(),"user_score"]
pc   = df.loc[(df["platform"]=="PC")   & df["user_score"].notna(),"user_score"]

# Welch t-test (varianzas no iguales)
stat1, p1 = st.ttest_ind(xone, pc, equal_var=False, nan_policy="omit") if (len(xone)>1 and len(pc)>1) else (np.nan, np.nan)

# H2: medias de user_score para Acción y Deportes son diferentes (H0: μ_Action = μ_Sports; H1: μ_Action ≠ μ_Sports)
action = df.loc[(df["genre"]=="Action")  & df["user_score"].notna(),"user_score"]
sports = df.loc[(df["genre"]=="Sports")  & df["user_score"].notna(),"user_score"]

stat2, p2 = st.ttest_ind(action, sports, equal_var=False, nan_policy="omit") if (len(action)>1 and len(sports)>1) else (np.nan, np.nan)

print(f"H1 XOne vs PC -> p-value: {p1:.6f}  | decisión:", "Rechazar H0" if (p1 < alpha) else "No rechazar H0")
print(f"H2 Action vs Sports -> p-value: {p2:.6f}  | decisión:", "Rechazar H0" if (p2 < alpha) else "No rechazar H0")




Formulación
- **H1 (plataformas):**  
  - H0: la media de `user_score` en **XOne** es igual a la de **PC**.  
  - H1: son distintas.  
  - **Criterio:** *t* de Welch (`equal_var=False`) por posibles varianzas distintas y tamaños desiguales.

- **H2 (géneros):**  
  - H0: la media de `user_score` en **Action** es igual a **Sports**.  
  - H1: son distintas.  
  - **Criterio:** *t* de Welch, bilateral (≠).

**Nivel de significancia:** α = 0.05.  
**Regla de decisión:** si `p-value < α`, rechazar H0.


# Paso 6. Escribe una conclusión general

- **Ventana de referencia:** el análisis se centra en **2013–2016**, periodo que refleja mejor el ciclo de vida actual de las consolas. Los datos de 2016 están incompletos, y los años anteriores a 2000 no son significativos.

- **Ciclo de vida de las plataformas:**  
  - Consolas como **PS2, Wii, DS y Xbox360** fueron muy populares en su momento, pero hoy prácticamente no generan ventas.  
  - Las plataformas nuevas (PS4, XOne) tardan **1–2 años** en despegar.  
  - Las antiguas suelen desaparecer en **2–3 años** tras la llegada de la nueva generación.  
  - El ciclo de vida total ronda los **10 años** (auge + declive).

- **Plataformas líderes (2013–2016):**  
  - **PS4** es la plataforma más prometedora, con ventas crecientes y mayor mediana por juego.  
  - **XOne** le sigue, con ventas altas pero menores a PS4.  
  - **3DS** mantiene peso relevante, sobre todo en Japón.  
  - **PS3 y X360** ya están en claro declive.

- **Distribución de ventas por juego:** los boxplots muestran colas largas → unos pocos “megahits” generan la mayoría de ingresos, mientras que la gran mayoría de juegos vende muy poco.

- **Reseñas y ventas (PS4):**  
  - **Critic_Score vs ventas**: correlación positiva moderada (≈0.41).  
  - **User_Score vs ventas**: correlación casi nula (≈-0.03).  
  - Implicación: las reseñas de críticos influyen más en ventas que las de usuarios → la cobertura en prensa especializada es clave.

- **Comparación multiplataforma:** los títulos disponibles en varias plataformas (ej. FIFA, Call of Duty, GTA V) concentran la mayoría de ventas en **PS4**, seguida de **XOne**. PC y portátiles suelen quedar muy por detrás.

- **Géneros más rentables:**  
  - **Action, Shooter y Sports** dominan a nivel global.  
  - **RPG** ocupa un lugar relevante, especialmente en Japón.  
  - Géneros como **Puzzle, Strategy o Adventure** muestran ventas bajas, más bien de nicho.  
  - La varianza y desviación estándar confirman que en géneros como Action hay mucha dispersión (algunos hits enormes y muchos juegos con ventas bajas).

- **Perfiles regionales:**  
  - **NA/EU:** consolas de sobremesa (PS4, XOne) + géneros Action/Shooter/Sports. Ratings ESRB **Mature** y **Teen** dominan las ventas.  
  - **JP:** portátiles (3DS, DS, PSP) + géneros RPG y Action. El rating ESRB tiene **poco impacto** (muchos juegos aparecen como “Unknown”).  

- **Pruebas de hipótesis (α=0.05):**  
  - **H1 (XOne vs PC en user_score):** p ≈ 0.000005 → **rechazamos H₀**. Las medias son diferentes.  
  - **H2 (Action vs Sports en user_score):** p ≈ 0.115 → **no se rechaza H₀**. No hay evidencia de diferencia en medias.

Finalmente se recomiendo que para 2017 Se hagan campañas principales en PS4 y XOne, enfocadas en generos de Action, Shooter y Sports en NA/EU. Para Japon, priorizar 3DS y géneros RPG, adaptando la estrategia local.
Por otro lado, dar mayor importancia a las reseñas de críticos, ya que son las que más influyen en ventas y finalmente anticipar que los juegos multiplataforma venderán mucho más en la consola con mayor base activa (PS4 en este periodo).  